In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

pd.set_option('display.max_columns', None)
%matplotlib inline

In [39]:
import pandas as pd
import requests
import requests

def download_from_drive(file_id, destination):
    url = "https://drive.google.com/uc?export=download"
    session = requests.Session()
    response = session.get(url, params={'id': file_id}, stream=True)

    token = None
    for key, value in response.cookies.items():
        if key.startswith('download_warning'):
            token = value

    if token:
        response = session.get(url, params={'id': file_id, 'confirm': token}, stream=True)

    with open(destination, 'wb') as f:
        for chunk in response.iter_content(32768):
            if chunk:
                f.write(chunk)

visits_id = "1QosQQ4RRNR9rkL4t7sB707h2Uy0XfYJe"
signups_id = "1AeQz0kaSgz0lxYSDtuNm36muhy5fRCzZ"

download_from_drive(visits_id, "visitas.csv")
download_from_drive(signups_id, "inscripciones.csv")

visits = pd.read_csv("visitas.csv")
signups = pd.read_csv("inscripciones.csv")

print("Visitas:", visits.shape)
print("Inscripciones:", signups.shape)

Visitas: (1000, 4)
Inscripciones: (1000, 5)


In [40]:
visits.head()

,uuid,platform,user_agent,date
0,1de9ea66-70d3-4a1f-8735-df5ef7697fb9,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-03-01T13:29:22
1,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-01T16:44:28
2,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-06T06:12:36
3,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-01T20:16:37
4,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-05T17:42:47


In [41]:
signups.head()

,date,user_id,email,platform,registration_type
0,2023-03-01T00:25:39,8838849,joseph95@example.org,web,google
1,2023-03-01T14:53:01,8741065,janetsuarez@example.net,web,yandex
2,2023-03-01T14:27:36,1866654,robert67@example.org,web,google
3,2023-03-01T02:42:34,1577584,elam@example.net,web,apple
4,2023-03-01T10:27:14,4765395,stephanie68@example.net,web,yandex


In [42]:
visits.describe(include='all')

,uuid,platform,user_agent,date
count,1000,1000,1000,1000
unique,519,3,28,996
top,251a0926-ece3-4d77-aa42-ab569fdf9fe2,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-03-01T08:01:45
freq,4,954,71,2


In [43]:
signups.describe(include='all')

,date,user_id,email,platform,registration_type
count,1000,1.000000e+03,1000,1000,1000
unique,1000,NaN,997,3,4
top,2023-03-01T00:25:39,NaN,zanderson@example.org,android,email
freq,1,NaN,2,517,446
mean,NaN,4.488623e+06,NaN,NaN,NaN
std,NaN,2.620568e+06,NaN,NaN,NaN
min,NaN,2.236800e+04,NaN,NaN,NaN
25%,NaN,2.235489e+06,NaN,NaN,NaN
50%,NaN,4.473044e+06,NaN,NaN,NaN
75%,NaN,6.779707e+06,NaN,NaN,NaN


In [44]:
visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   uuid        1000 non-null   object
 1   platform    1000 non-null   object
 2   user_agent  1000 non-null   object
 3   date        1000 non-null   object
dtypes: object(4)
memory usage: 31.4+ KB


In [45]:
signups.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   date               1000 non-null   object
 1   user_id            1000 non-null   int64 
 2   email              1000 non-null   object
 3   platform           1000 non-null   object
 4   registration_type  1000 non-null   object
dtypes: int64(1), object(4)
memory usage: 39.2+ KB


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

API_URL = os.getenv('API_URL')
DATE_BEGIN = os.getenv('DATE_BEGIN')
DATE_END = os.getenv('DATE_END')

api_visits_url = f"{API_URL}/visits"
api_registrations_url = f"{API_URL}/registrations"

params = {
    "begin": DATE_BEGIN,
    "end": DATE_END
}

response_visits = requests.get(api_visits_url, params=params)
response_registrations = requests.get(api_registrations_url, params=params)

print("Status visitas:", response_visits.status_code)
print("Status registros:", response_registrations.status_code)


In [ ]:
api_visits = pd.DataFrame(response_visits.json())
api_registrations = pd.DataFrame(response_registrations.json())

print("API Visitas:", api_visits.shape)
print("API Registros:", api_registrations.shape)

In [ ]:
api_visits.head()

In [ ]:
api_registrations.head()

In [ ]:
api_visits['datetime'] = pd.to_datetime(api_visits['datetime'])
api_registrations['datetime'] = pd.to_datetime(api_registrations['datetime'])

# Excluir bots: los que tengan "bot" en platform o en user_agent
is_bot = (
    api_visits['platform'].str.contains('bot', case=False, na=False) |
    api_visits['user_agent'].str.contains('bot', case=False, na=False)
)
visits_clean = api_visits[~is_bot].copy()

# Quedarnos con la ÚLTIMA visita de cada visit_id
visits_clean = visits_clean.sort_values('datetime').drop_duplicates(subset='visit_id', keep='last')

print("Visitas totales:", len(api_visits))
print("Visitas sin bots ni duplicados:", len(visits_clean))

In [ ]:
visits_clean['date_group'] = visits_clean['datetime'].dt.floor('D')

visits_grouped = (
    visits_clean.groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='visits')
)
visits_grouped.head()

In [ ]:
api_registrations['date_group'] = api_registrations['datetime'].dt.floor('D')

registrations_grouped = (
    api_registrations.groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='registrations')
)
registrations_grouped.head()

In [ ]:
conversion = pd.merge(
    visits_grouped, registrations_grouped,
    on=['date_group', 'platform'], how='outer'
)
conversion['visits'] = conversion['visits'].fillna(0)
conversion['registrations'] = conversion['registrations'].fillna(0)

conversion['conversion'] = (conversion['registrations'] / conversion['visits']) * 100

conversion = conversion.sort_values(['date_group', 'platform']).reset_index(drop=True)

conversion.head()

In [ ]:
conversion.to_json("./conversion.json")
print("Archivo guardado ✅")

In [ ]:
ads_id = "12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj"
download_from_drive(ads_id, "ads.csv")

# Revisamos si el archivo trae encabezado o no
with open("ads.csv") as f:
    for _ in range(3):
        print(f.readline())

In [ ]:
ads_columns = ['date', 'utm_source', 'utm_medium', 'utm_campaign', 'cost']

first_line = open('ads.csv').readline()
if first_line.lower().startswith('date'):
    ads = pd.read_csv('ads.csv')
else:
    ads = pd.read_csv('ads.csv', header=None, names=ads_columns)

ads['date'] = pd.to_datetime(ads['date'])
print(ads.shape)
ads.head()

In [ ]:
ads['date_group'] = ads['date'].dt.floor('D')

ads_grouped = (
    ads.groupby('date_group')
    .agg(
        cost=('cost', 'sum'),
        utm_campaign=('utm_campaign', lambda x: ', '.join(sorted(set(x))))
    )
    .reset_index()
)
ads_grouped.head()

In [ ]:
daily_totals = conversion.groupby('date_group')[['visits', 'registrations']].sum().reset_index()

ads_final = pd.merge(daily_totals, ads_grouped, on='date_group', how='left')

ads_final['cost'] = ads_final['cost'].fillna(0)
ads_final['utm_campaign'] = ads_final['utm_campaign'].fillna('none')

ads_final = ads_final.sort_values('date_group').reset_index(drop=True)


In [ ]:
ads_final.to_json("./ads.json")
print("Archivo guardado ✅")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

import os
os.makedirs('./charts', exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
visitas_totales = conversion.groupby('date_group')['visits'].sum().reset_index()

plt.figure()
plt.plot(visitas_totales['date_group'], visitas_totales['visits'], color='steelblue')
plt.title('Visitas totales por día')
plt.xlabel('Fecha')
plt.ylabel('Visitas')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/total_visits.png')
plt.show()

In [ ]:
plt.figure()
for plat in conversion['platform'].unique():
    data = conversion[conversion['platform'] == plat]
    plt.plot(data['date_group'], data['visits'], label=plat)
plt.title('Visitas por plataforma')
plt.xlabel('Fecha')
plt.ylabel('Visitas')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/visits_by_platform.png')
plt.show()

In [ ]:
registros_totales = conversion.groupby('date_group')['registrations'].sum().reset_index()

plt.figure()
plt.plot(registros_totales['date_group'], registros_totales['registrations'], color='darkorange')
plt.title('Registros totales por día')
plt.xlabel('Fecha')
plt.ylabel('Registros')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/total_registrations.png')
plt.show()

In [ ]:
plt.figure()
for plat in conversion['platform'].unique():
    data = conversion[conversion['platform'] == plat]
    plt.plot(data['date_group'], data['registrations'], label=plat)
plt.title('Registros por plataforma')
plt.xlabel('Fecha')
plt.ylabel('Registros')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/registrations_by_platform.png')
plt.show()

In [ ]:
conv_by_platform = conversion.groupby('platform').apply(
    lambda x: (x['registrations'].sum() / x['visits'].sum()) * 100
).reset_index(name='conversion')

plt.figure()
sns.barplot(data=conv_by_platform, x='platform', y='conversion', palette='viridis')
plt.title('Conversión promedio por plataforma')
plt.xlabel('Plataforma')
plt.ylabel('Conversión (%)')
plt.tight_layout()
plt.savefig('./charts/conversion_by_platform.png')
plt.show()

In [ ]:
conv_daily = conversion.groupby('date_group').apply(
    lambda x: (x['registrations'].sum() / x['visits'].sum()) * 100
).reset_index(name='conversion')

plt.figure()
plt.plot(conv_daily['date_group'], conv_daily['conversion'], color='seagreen')
plt.axhline(conv_daily['conversion'].mean(), color='red', linestyle='--', label=f"Promedio: {conv_daily['conversion'].mean():.1f}%")
plt.title('Conversión promedio diaria')
plt.xlabel('Fecha')
plt.ylabel('Conversión (%)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/average_conversion.png')
plt.show()

In [ ]:
plt.figure()
plt.bar(ads_final['date_group'], ads_final['cost'], color='indianred')
plt.title('Costos diarios de campañas publicitarias')
plt.xlabel('Fecha')
plt.ylabel('Costo')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/ads_cost.png')
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
campaigns = ads_final['utm_campaign'].unique()
palette = sns.color_palette('tab10', len(campaigns))
color_map = dict(zip(campaigns, palette))
colors = ads_final['utm_campaign'].map(color_map)

plt.bar(ads_final['date_group'], ads_final['visits'], color=colors)
handles = [plt.Rectangle((0,0),1,1, color=color_map[c]) for c in campaigns]
plt.legend(handles, campaigns, title='Campaña', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.title('Visitas por período según campaña activa')
plt.xlabel('Fecha')
plt.ylabel('Visitas')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/visits_by_campaign.png')
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.bar(ads_final['date_group'], ads_final['registrations'], color=colors)
plt.legend(handles, campaigns, title='Campaña', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.title('Registros por período según campaña activa')
plt.xlabel('Fecha')
plt.ylabel('Registros')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/registrations_by_campaign.png')
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14,6))

ax1.bar(ads_final['date_group'], ads_final['cost'], color='lightgray', label='Gasto publicitario', alpha=0.6)
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Gasto publicitario ($)', color='gray')
ax1.tick_params(axis='y', labelcolor='gray')

ax2 = ax1.twinx()
ax2.plot(ads_final['date_group'], ads_final['visits'], color='blue', label='Visitas')
ax2.plot(ads_final['date_group'], ads_final['registrations'], color='green', label='Registros')
ax2.set_ylabel('Visitas / Registros')

fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.title('Visitas y registros vs. gasto publicitario en el tiempo')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/visits_registrations_vs_cost.png')
plt.show()

In [ ]:
daily_conversion = conversion.groupby('date_group').agg({'visits':'sum','registrations':'sum'}).reset_index()
daily_conversion['conversion_rate'] = (daily_conversion['registrations'] / daily_conversion['visits']) * 100

avg_rate = daily_conversion['conversion_rate'].mean()

plt.figure(figsize=(14,6))
plt.plot(daily_conversion['date_group'], daily_conversion['conversion_rate'], label='Tasa de conversión diaria')
plt.axhline(avg_rate, color='red', linestyle='--', label=f'Promedio: {avg_rate:.2f}%')
plt.title('Tasa de conversión diaria')
plt.xlabel('Fecha')
plt.ylabel('Conversión (%)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('./charts/daily_conversion_rate.png')
plt.show()

In [ ]:
daily_conversion['visits_pct_change'] = daily_conversion['visits'].pct_change() * 100
daily_conversion['registrations_pct_change'] = daily_conversion['registrations'].pct_change() * 100

print("Top 5 caídas en VISITAS:")
print(daily_conversion.nsmallest(5, 'visits_pct_change')[['date_group', 'visits', 'visits_pct_change']])

print("\nTop 5 caídas en REGISTROS:")
print(daily_conversion.nsmallest(5, 'registrations_pct_change')[['date_group', 'registrations', 'registrations_pct_change']])

In [ ]:
corr_data = ads_final[['cost', 'visits', 'registrations']].corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr_data, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlación entre gasto, visitas y registros')
plt.tight_layout()
plt.savefig('./charts/correlation_matrix.png')
plt.show()

## 📊 Presentación de resultados: Análisis de conversión y publicidad

### Situación
Se analizó el período 2023-03-01 a 2023-09-01 (6 meses), cruzando datos de tráfico web, registros de usuarios y gasto en campañas publicitarias, para evaluar el impacto de la inversión publicitaria y detectar anomalías en el embudo de conversión.

### Hallazgos

**1. Relación entre publicidad y tráfico/registros**
- La correlación entre gasto (`cost`) y visitas es de 0.30, y entre gasto y registros es de 0.31: una relación positiva pero débil.
- Esto indica que aumentar el presupuesto publicitario acompaña cierto incremento de tráfico y registros, pero está lejos de ser el principal factor.
- La correlación entre visitas y registros es más alta (0.58): más tráfico tiende a producir más registros, aunque la tasa de conversión diaria no es constante (osciló entre ~7% y ~28%, con un promedio de 16.49%).

**2. Caídas en visitas (top 5)**
| Fecha | Visitas | Caída |
|---|---|---|
| 2023-03-16 | 394 | -67.1% |
| 2023-08-16 | 471 | -58.5% |
| 2023-07-27 | 362 | -52.6% |
| 2023-04-14 | 544 | -50.5% |
| 2023-05-01 | 453 | -49.4% |

**3. Caídas en registros (top 5)**
| Fecha | Registros | Caída |
|---|---|---|
| 2023-05-01 | 59 | -66.5% |
| 2023-06-02 | 48 | -61.3% |
| 2023-08-16 | 75 | -52.8% |
| 2023-08-27 | 88 | -52.4% |
| 2023-06-20 | 50 | -49.0% |

**4. Cruce entre ambas listas**
Dos fechas (2023-05-01 y 2023-08-16) aparecen en ambas listas de caídas: en esos días el problema parece originarse en una baja de tráfico que arrastró también los registros (posible pausa o cambio en las campañas ese día).

En cambio, 2023-06-02, 2023-06-20 y 2023-08-27 muestran caídas fuertes en registros **sin** una caída equivalente en visitas: esto sugiere que el tráfico se mantuvo relativamente estable esos días, pero algo falló en el proceso de registro en sí (posible problema técnico, de formulario, o de calidad del tráfico atraído esos días).

### Recomendación
- **Investigar puntualmente** los días 2023-06-02, 2023-06-20 y 2023-08-27 con el equipo técnico/producto, revisando logs del formulario de registro o cambios de código desplegados esas fechas, ya que ahí la caída no se explica por el tráfico.
- **Revisar el calendario de campañas** alrededor del 2023-05-01 y 2023-08-16, para confirmar si coincidieron con pausas de presupuesto o cambios de creativos.
- **No aumentar el presupuesto publicitario a ciegas**: dado que la correlación gasto-registros es débil (0.31), subir la inversión sin optimizar antes el embudo de conversión (16.49% promedio) tendría un retorno limitado.
- **Mantener las campañas que sí sostienen tráfico consistente**, visibles en el gráfico de doble eje durante jun-jul y ago-sep, donde mayor gasto coincidió con visitas más altas y estables.

### Impacto esperado
- Corregir los problemas técnicos en los días de caída aislada de registros podría recuperar entre 50% y 65% de los registros perdidos en esas fechas puntuales: un efecto inmediato y de bajo costo.
- Optimizar la conversión antes de escalar el gasto publicitario permitiría capturar más registros por cada peso invertido, mejorando el retorno de la inversión sin necesariamente aumentar el presupuesto.